# Transformer pipeline (preprocessing)

Этот ноутбук начинает пайплайн с разметки сплитов и подготовки light-curve данных для трансформера.


In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path


## Разметка сплитов

Берем тот же split-стратегию, что и в `pipeline.ipynb`, чтобы избежать утечки между train/val.


In [ ]:
DATA_ROOT = Path("../datasets/")

@dataclass
class Config:
    seed: int = 42
    n_folds: int = 5

def load_meta():
    train_log = pd.read_csv(DATA_ROOT / "train_log.csv")
    test_log = pd.read_csv(DATA_ROOT / "test_log.csv")
    return train_log, test_log

def get_splits(train_log: pd.DataFrame, n_folds: int = 5):
    splits = sorted(train_log["split"].unique())
    folds = []
    for i in range(n_folds):
        folds.append(splits[i::n_folds])
    return folds

def get_fold_indices(train_log: pd.DataFrame, val_splits: list[str]):
    is_val = train_log["split"].isin(val_splits)
    train_idx = train_log.index[~is_val].to_numpy()
    val_idx = train_log.index[is_val].to_numpy()
    return train_idx, val_idx


In [ ]:
train_log, test_log = load_meta()
folds = get_splits(train_log, n_folds=5)
print('folds:', folds)


## Загрузка и очистка light curves


In [ ]:
class Curver:
    def load_lightcurves_for_splits(self, splits: list[str], kind: str) -> pd.DataFrame:
        parts = []
        for split in splits:
            fname = 'train_full_lightcurves.csv' if kind == 'train' else 'test_full_lightcurves.csv'
            path = DATA_ROOT / split / fname
            df = pd.read_csv(path)
            df["splits"] = split
            parts.append(df)
        return pd.concat(parts, ignore_index=True)

    def clean_lightcurves(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc = lc[lc["Flux"].notna()]
        lc["Time (MJD)"] = lc["Time (MJD)"].astype(float)
        lc["Flux"] = lc["Flux"].astype(float)
        lc["Flux_err"] = lc["Flux_err"].astype(float)
        return lc

    def add_time_features(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc["t0"] = lc.groupby("object_id")["Time (MJD)"].transform("min")
        lc["dt"] = lc["Time (MJD)"] - lc["t0"]
        return lc


## Опциональная фильтрация по минимальному числу наблюдений

Для трансформера удобно убрать слишком короткие кривые (например, < 20 точек). Это нужно делать отдельно для train/val, чтобы не было leakage.


In [ ]:
def filter_objects_by_min_obs(lc: pd.DataFrame, min_obs: int = 20):
    counts = lc.groupby("object_id")["Flux"].count()
    keep_ids = counts[counts >= min_obs].index
    lc_filt = lc[lc["object_id"].isin(keep_ids)].copy()
    return lc_filt, set(keep_ids)


In [ ]:
curve = Curver()

val_splits = folds[0]
train_splits = [s for s in sorted(train_log["split"].unique()) if s not in val_splits]

train_lc = curve.load_lightcurves_for_splits(train_splits, kind="train")
val_lc = curve.load_lightcurves_for_splits(val_splits, kind="train")

train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))
val_lc = curve.add_time_features(curve.clean_lightcurves(val_lc))

train_lc, train_keep = filter_objects_by_min_obs(train_lc, min_obs=20)
val_lc, val_keep = filter_objects_by_min_obs(val_lc, min_obs=20)

train_meta = train_log[train_log["object_id"].isin(train_keep)].copy()
val_meta = train_log[train_log["object_id"].isin(val_keep)].copy()

print('train objects:', train_meta.shape[0])
print('val objects:', val_meta.shape[0])


Дальше: формирование последовательностей фиксированной длины (padding + mask), создание Dataset/Loader и обучение трансформера.


## Формирование последовательностей для трансформера

Каждому объекту строим последовательность наблюдений `T x F` с padding и mask.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


In [ ]:
FILTERS = ['u', 'g', 'r', 'i', 'z', 'y']
FILTER_TO_IDX = {f: i for i, f in enumerate(FILTERS)}

In [ ]:
def _robust_zscore(x: np.ndarray):
    x = x.astype(float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    scale = mad if mad > 0 else np.nanstd(x)
    if not np.isfinite(scale) or scale == 0:
        return np.full_like(x, 0.0, dtype=float)
    return (x - med) / scale

def build_sequence_for_object(grp: pd.DataFrame, max_len: int):
    grp = grp.sort_values('dt')

    dt = grp['dt'].to_numpy(dtype=float)
    flux = grp['Flux'].to_numpy(dtype=float)
    ferr = grp['Flux_err'].to_numpy(dtype=float)
    fidx = grp['Filter'].map(FILTER_TO_IDX).fillna(-1).to_numpy(dtype=int)

    flux_z = _robust_zscore(flux)
    snr = np.where(ferr > 0, flux / ferr, 0.0)

    feats = np.stack([dt, flux, ferr, flux_z, snr], axis=1)

    if len(feats) >= max_len:
        feats = feats[:max_len]
        fidx = fidx[:max_len]
        mask = np.ones(max_len, dtype=bool)
    else:
        pad = max_len - len(feats)
        feats = np.pad(feats, ((0, pad), (0, 0)), mode='constant', constant_values=0.0)
        fidx = np.pad(fidx, (0, pad), mode='constant', constant_values=-1)
        mask = np.zeros(max_len, dtype=bool)
        mask[:len(grp)] = True

    return feats, fidx, mask

In [ ]:
def build_sequences(lc: pd.DataFrame, meta: pd.DataFrame, max_len: int = 256):
    feats_list = []
    filt_list = []
    mask_list = []
    labels = []
    object_ids = []

    for obj_id, grp in lc.groupby('object_id'):
        feats, fidx, mask = build_sequence_for_object(grp, max_len=max_len)
        feats_list.append(feats)
        filt_list.append(fidx)
        mask_list.append(mask)
        object_ids.append(obj_id)

    seq_feats = np.stack(feats_list)
    seq_filters = np.stack(filt_list)
    seq_mask = np.stack(mask_list)

    meta_idx = meta.set_index('object_id')
    labels = meta_idx.loc[object_ids]['target'].astype(int).to_numpy()

    return seq_feats, seq_filters, seq_mask, labels, np.array(object_ids)


In [ ]:
class LightcurveDataset(Dataset):
    def __init__(self, seq_feats, seq_filters, seq_mask, labels, object_ids):
        self.seq_feats = torch.tensor(seq_feats, dtype=torch.float32)
        self.seq_filters = torch.tensor(seq_filters, dtype=torch.long)
        self.seq_mask = torch.tensor(seq_mask, dtype=torch.bool)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.object_ids = object_ids

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'x': self.seq_feats[idx],
            'filter_idx': self.seq_filters[idx],
            'mask': self.seq_mask[idx],
            'y': self.labels[idx],
            'object_id': self.object_ids[idx],
        }


## Подготовка Dataset/DataLoader


In [ ]:
MAX_LEN = 256

train_seq, train_filt, train_mask, train_y, train_ids = build_sequences(
    train_lc, train_meta, max_len=MAX_LEN
)
val_seq, val_filt, val_mask, val_y, val_ids = build_sequences(
    val_lc, val_meta, max_len=MAX_LEN
)

train_ds = LightcurveDataset(train_seq, train_filt, train_mask, train_y, train_ids)
val_ds = LightcurveDataset(val_seq, val_filt, val_mask, val_y, val_ids)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, drop_last=False)

batch = next(iter(train_loader))
print('batch x:', batch['x'].shape)
print('batch filter_idx:', batch['filter_idx'].shape)
print('batch mask:', batch['mask'].shape)
print('batch y:', batch['y'].shape)


Дальше: определение модели трансформера, позиционного кодирования и цикла обучения.


## Архитектура трансформера

Модель принимает последовательность `T x F`, добавляет embedding фильтра и позиционное кодирование по `dt`. Дальше несколько encoder-блоков и pooling по маске.


In [15]:
import torch.nn as nn

In [16]:
class Time2Vec(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.linear = nn.Linear(1, 1)
        self.periodic = nn.Linear(1, d_model - 1)

    def forward(self, t):
        t = t.unsqueeze(-1)
        lin = self.linear(t)
        per = torch.sin(self.periodic(t))
        return torch.cat([lin, per], dim=-1)

In [17]:
class TransformerClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_filters: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.filter_emb = nn.Embedding(n_filters + 1, d_model)
        self.input_proj = nn.Linear(input_dim, d_model)
        self.time_enc = Time2Vec(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x, filter_idx, mask):
        dt = x[..., 0]
        x_proj = self.input_proj(x)

        fidx = torch.clamp(filter_idx, min=0, max=self.filter_emb.num_embeddings - 1)
        f_emb = self.filter_emb(fidx)

        t_enc = self.time_enc(dt)
        h = x_proj + f_emb + t_enc

        key_padding_mask = ~mask
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)

        h = self.norm(h)
        mask_f = mask.unsqueeze(-1).float()
        pooled = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        logits = self.head(pooled).squeeze(-1)
        return logits


In [18]:
model = TransformerClassifier(
    input_dim=train_seq.shape[-1],
    n_filters=len(FILTERS),
    d_model=128,
    n_heads=4,
    n_layers=4,
    dropout=0.1,
)
print(model)


TransformerClassifier(
  (filter_emb): Embedding(7, 128)
  (input_proj): Linear(in_features=5, out_features=128, bias=True)
  (time_enc): Time2Vec(
    (linear): Linear(in_features=1, out_features=1, bias=True)
    (periodic): Linear(in_features=1, out_features=127, bias=True)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): L

#### Количество параметров модели

In [19]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_params(model)
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,}")

Total params: 2,390,913
Trainable params: 2,390,913


## Обучение модели

Логируем loss и метрики на train/val. Для оценки используем F1 и подбираем порог на валидации.


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


In [ ]:
def f1_score_binary(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else 2 * tp / denom

def best_f1_threshold_np(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_thr = 0.5
    best_f1 = -1.0
    for thr in thresholds:
        f1 = f1_score_binary(y_true, y_prob >= thr)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return best_thr, best_f1


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        y = batch['y'].to(device)

        optimizer.zero_grad()
        logits = model(x, fidx, mask)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_true = []
    for batch in loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        y = batch['y'].to(device)

        logits = model(x, fidx, mask)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)

        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_true.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_true = np.concatenate(all_true)
    thr, best_f1 = best_f1_threshold_np(all_true, all_probs)
    f1_default = f1_score_binary(all_true, all_probs >= 0.5)

    metrics = {
        'val_loss': total_loss / len(loader.dataset),
        'f1@0.5': f1_default,
        'best_f1': best_f1,
        'best_thr': thr,
    }
    return metrics


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

pos = (train_y == 1).sum()
neg = (train_y == 0).sum()
pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['val_loss']:.4f} | "
        f"f1@0.5={val_metrics['f1@0.5']:.4f} | "
        f"best_f1={val_metrics['best_f1']:.4f} | "
        f"best_thr={val_metrics['best_thr']:.2f}"
    )
    scheduler.step()


## Метрики AUC/PR-AUC, сохранение best model, early stopping


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
@torch.no_grad()
def evaluate_with_auc(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_true = []
    for batch in loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        y = batch['y'].to(device)

        logits = model(x, fidx, mask)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)

        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_true.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_true = np.concatenate(all_true)

    thr, best_f1 = best_f1_threshold_np(all_true, all_probs)
    f1_default = f1_score_binary(all_true, all_probs >= 0.5)

    auc = roc_auc_score(all_true, all_probs) if len(np.unique(all_true)) > 1 else 0.5
    pr_auc = average_precision_score(all_true, all_probs) if len(np.unique(all_true)) > 1 else 0.0

    metrics = {
        'val_loss': total_loss / len(loader.dataset),
        'f1@0.5': f1_default,
        'best_f1': best_f1,
        'best_thr': thr,
        'auc': auc,
        'pr_auc': pr_auc,
    }
    return metrics


In [ ]:
BEST_MODEL_PATH = 'best_transformer.pt'
PATIENCE = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

pos = (train_y == 1).sum()
neg = (train_y == 0).sum()
pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 20
best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate_with_auc(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['val_loss']:.4f} | "
        f"f1@0.5={val_metrics['f1@0.5']:.4f} | "
        f"best_f1={val_metrics['best_f1']:.4f} | "
        f"best_thr={val_metrics['best_thr']:.2f} | "
        f"auc={val_metrics['auc']:.4f} | "
        f"pr_auc={val_metrics['pr_auc']:.4f}"
    )

    if val_metrics['best_f1'] > best_f1:
        best_f1 = val_metrics['best_f1']
        bad_epochs = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"Saved best model to {BEST_MODEL_PATH} (best_f1={best_f1:.4f})")
    else:
        bad_epochs += 1
        print(f"No improvement. bad_epochs={bad_epochs}/{PATIENCE}")

    if bad_epochs >= PATIENCE:
        print('Early stopping triggered.')
        break

    scheduler.step()


## OOF Cross-Validation по сплитам

Считаем OOF предсказания по всем фолдам, подбираем общий порог и сохраняем лучшие веса каждого фолда.


In [ ]:
def train_one_fold_transformer(train_splits, val_splits, fold_id, max_len=256):
    train_meta = train_log[train_log['split'].isin(train_splits)].copy()
    val_meta = train_log[train_log['split'].isin(val_splits)].copy()

    train_lc = curve.load_lightcurves_for_splits(train_splits, kind='train')
    val_lc = curve.load_lightcurves_for_splits(val_splits, kind='train')
    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))
    val_lc = curve.add_time_features(curve.clean_lightcurves(val_lc))

    train_lc, train_keep = filter_objects_by_min_obs(train_lc, min_obs=20)
    val_lc, val_keep = filter_objects_by_min_obs(val_lc, min_obs=20)
    train_meta = train_meta[train_meta['object_id'].isin(train_keep)].copy()
    val_meta = val_meta[val_meta['object_id'].isin(val_keep)].copy()

    train_seq, train_filt, train_mask, train_y, train_ids = build_sequences(
        train_lc, train_meta, max_len=max_len
    )
    val_seq, val_filt, val_mask, val_y, val_ids = build_sequences(
        val_lc, val_meta, max_len=max_len
    )

    train_ds = LightcurveDataset(train_seq, train_filt, train_mask, train_y, train_ids)
    val_ds = LightcurveDataset(val_seq, val_filt, val_mask, val_y, val_ids)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, drop_last=False)

    model = TransformerClassifier(
        input_dim=train_seq.shape[-1],
        n_filters=len(FILTERS),
        d_model=128,
        n_heads=4,
        n_layers=4,
        dropout=0.1,
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    pos = (train_y == 1).sum()
    neg = (train_y == 0).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=10)

    best_f1 = -1.0
    bad_epochs = 0
    best_path = f'best_transformer_fold{fold_id}.pt'

    for epoch in range(1, 21):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate_with_auc(model, val_loader, criterion, device)

        print(
            f"Fold {fold_id} | Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['val_loss']:.4f} | "
            f"best_f1={val_metrics['best_f1']:.4f} | "
            f"auc={val_metrics['auc']:.4f} | "
            f"pr_auc={val_metrics['pr_auc']:.4f}"
        )

        if val_metrics['best_f1'] > best_f1:
            best_f1 = val_metrics['best_f1']
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
        else:
            bad_epochs += 1

        if bad_epochs >= 3:
            break

        scheduler.step()

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    all_probs = []
    all_true = []
    for batch in val_loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        y = batch['y'].to(device)
        logits = model(x, fidx, mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_true.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_true = np.concatenate(all_true)
    return all_probs, all_true, val_ids, best_path


In [ ]:
all_splits = sorted(train_log['split'].unique())
folds = get_splits(train_log, n_folds=5)

oof_pred = np.full(len(train_log), np.nan)
oof_true = train_log['target'].astype(int).to_numpy()
id_to_idx = {oid: i for i, oid in enumerate(train_log['object_id'].values)}

fold_paths = []
for fold_id, val_splits in enumerate(folds):
    train_splits = [s for s in all_splits if s not in val_splits]
    val_probs, val_true, val_ids, best_path = train_one_fold_transformer(
        train_splits, val_splits, fold_id, max_len=MAX_LEN
    )
    fold_paths.append(best_path)

    for oid, p in zip(val_ids, val_probs):
        oof_pred[id_to_idx[oid]] = p

mask = np.isfinite(oof_pred)
oof_thr, oof_f1 = best_f1_threshold_np(oof_true[mask], oof_pred[mask])
print(f'OOF best_f1={oof_f1:.4f}, thr={oof_thr:.2f}')


## Финальная тренировка и предсказание теста


In [ ]:
def train_full_transformer(max_len=256):
    all_splits = sorted(train_log['split'].unique())
    train_lc = curve.load_lightcurves_for_splits(all_splits, kind='train')
    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))

    train_lc, train_keep = filter_objects_by_min_obs(train_lc, min_obs=20)
    train_meta = train_log[train_log['object_id'].isin(train_keep)].copy()

    train_seq, train_filt, train_mask, train_y, train_ids = build_sequences(
        train_lc, train_meta, max_len=max_len
    )

    train_ds = LightcurveDataset(train_seq, train_filt, train_mask, train_y, train_ids)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)

    model = TransformerClassifier(
        input_dim=train_seq.shape[-1],
        n_filters=len(FILTERS),
        d_model=128,
        n_heads=4,
        n_layers=4,
        dropout=0.1,
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    pos = (train_y == 1).sum()
    neg = (train_y == 0).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=10)

    best_f1 = -1.0
    best_path = 'best_transformer_full.pt'
    for epoch in range(1, 21):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        print(f'Full | Epoch {epoch:02d} | train_loss={train_loss:.4f}')
        if train_loss < best_f1 or best_f1 < 0:
            best_f1 = train_loss
            torch.save(model.state_dict(), best_path)
        scheduler.step()

    model.load_state_dict(torch.load(best_path, map_location=device))
    return model, device

def predict_test_transformer(model, device, max_len=256):
    all_splits = sorted(test_log['split'].unique())
    test_lc = curve.load_lightcurves_for_splits(all_splits, kind='test')
    test_lc = curve.add_time_features(curve.clean_lightcurves(test_lc))

    test_seq, test_filt, test_mask, _, test_ids = build_sequences(
        test_lc, test_log, max_len=max_len
    )

    test_ds = LightcurveDataset(test_seq, test_filt, test_mask, np.zeros(len(test_ids)), test_ids)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, drop_last=False)

    model.eval()
    all_probs = []
    for batch in test_loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        logits = model(x, fidx, mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)

    all_probs = np.concatenate(all_probs)
    return test_ids, all_probs


In [ ]:
final_thr = float(oof_thr) if 'oof_thr' in globals() else 0.5

final_model, device = train_full_transformer(max_len=MAX_LEN)
test_ids, test_probs = predict_test_transformer(final_model, device, max_len=MAX_LEN)

test_pred = (test_probs >= final_thr).astype(int)
submission = pd.DataFrame({'object_id': test_ids, 'prediction': test_pred})
submission.to_csv('submission_transformer.csv', index=False)
print('Saved submission_transformer.csv, positives:', int(test_pred.sum()))
